# LoRA vs DoRA — Text-to-SQL on Spider

Compares LoRA and DoRA at rank 8 on a text-to-SQL task, measuring execution accuracy on the Spider dev set. `USE_DORA` in the config cell below is the only thing that changes between the two runs.

## 0. Setup

In [ ]:
!pip install -q -U transformers accelerate peft bitsandbytes trl datasets evaluate huggingface_hub

In [ ]:
from huggingface_hub import login
login()  # paste HF token when prompted

## 1. Config

In [ ]:
RANK = 8
USE_DORA = False      # False = LoRA, True = DoRA
SEED = 42

MODEL_NAME = "Qwen/Qwen2.5-1.5B"
TRAIN_SUBSET_SIZE = 3000
MAX_SEQ_LEN = 1024
OUTPUT_DIR = f"./results/rank{RANK}_{'dora' if USE_DORA else 'lora'}_seed{SEED}"
RUN_NAME = f"rank{RANK}_{'dora' if USE_DORA else 'lora'}_seed{SEED}"

print(f"Running: {RUN_NAME}")

## 2. Load and format Spider dataset

In [ ]:
from datasets import load_dataset
import random

random.seed(SEED)

spider = load_dataset("xlangai/spider")
print(spider)
print(spider["train"][0])

In [ ]:
# schema text isn't in this dataset version -- build it from spider's tables.json
import json, os, urllib.request

TABLES_JSON_URL = "https://raw.githubusercontent.com/taoyds/spider/master/tables.json"
if not os.path.exists("tables.json"):
    urllib.request.urlretrieve(TABLES_JSON_URL, "tables.json")

with open("tables.json") as f:
    tables_raw = json.load(f)

schema_by_db = {}
for db in tables_raw:
    db_id = db["db_id"]
    table_names = db["table_names_original"]
    col_names = db["column_names_original"]
    cols_by_table = {i: [] for i in range(len(table_names))}
    for tbl_idx, col_name in col_names:
        if tbl_idx == -1:
            continue
        cols_by_table[tbl_idx].append(col_name)
    schema_lines = []
    for i, tname in enumerate(table_names):
        schema_lines.append(f"{tname}({', '.join(cols_by_table[i])})")
    schema_by_db[db_id] = " | ".join(schema_lines)

print(list(schema_by_db.items())[0])

In [ ]:
PROMPT_TEMPLATE = """### Schema:
{schema}

### Question:
{question}

### SQL:
{sql}"""

def format_example(ex):
    schema = schema_by_db.get(ex["db_id"], "")
    text = PROMPT_TEMPLATE.format(schema=schema, question=ex["question"], sql=ex["query"])
    return {"text": text}

train_data = spider["train"].shuffle(seed=SEED).select(range(min(TRAIN_SUBSET_SIZE, len(spider["train"]))))
train_data = train_data.map(format_example)

eval_data = spider["validation"]

print(train_data[0]["text"])

## 3. Load model (4-bit) + tokenizer

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
model.config.use_cache = False

## 4. PEFT config — LoRA vs DoRA is one flag here

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=RANK,
    lora_alpha=RANK * 2,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    use_dora=USE_DORA,
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

## 5. Training

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    run_name=RUN_NAME,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    save_strategy="no",
    bf16=True,
    seed=SEED,
    max_length=MAX_SEQ_LEN,
    dataset_text_field="text",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    processing_class=tokenizer,
)

train_result = trainer.train()

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

## 6. Eval — execution accuracy on Spider dev set

In [ ]:
# xlangai/spider dropped its sqlite database files -- this mirror still has them
from huggingface_hub import snapshot_download

spider_db_snapshot = snapshot_download(
    repo_id="prem-research/spider",
    repo_type="dataset",
    allow_patterns=["database/*"],
)

SPIDER_DB_DIR = os.path.join(spider_db_snapshot, "database")
print(SPIDER_DB_DIR)
print(os.listdir(SPIDER_DB_DIR)[:10])

In [ ]:
import sqlite3

def run_sql(db_id, sql):
    db_path = os.path.join(SPIDER_DB_DIR, db_id, f"{db_id}.sqlite")
    try:
        conn = sqlite3.connect(db_path)
        cur = conn.cursor()
        cur.execute(sql)
        result = cur.fetchall()
        conn.close()
        return set(map(tuple, result))
    except Exception:
        return None  # bad sql counts as wrong

def generate_sql(question, db_id, max_new_tokens=128):
    schema = schema_by_db.get(db_id, "")
    prompt = f"### Schema:\n{schema}\n\n### Question:\n{question}\n\n### SQL:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    decoded = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return decoded.split("\n")[0].strip()

def execution_accuracy(eval_dataset, n=None):
    n = n or len(eval_dataset)
    correct = 0
    total = 0
    for i in range(n):
        ex = eval_dataset[i]
        pred_sql = generate_sql(ex["question"], ex["db_id"])
        pred_result = run_sql(ex["db_id"], pred_sql)
        gold_result = run_sql(ex["db_id"], ex["query"])
        total += 1
        if pred_result is not None and pred_result == gold_result:
            correct += 1
    return correct / total if total else 0.0

acc = execution_accuracy(eval_data, n=300)
print(f"{RUN_NAME} execution accuracy: {acc:.4f}")

## 7. Log result

In [ ]:
import csv

RESULTS_CSV = "results_log.csv"
row = {
    "run_name": RUN_NAME,
    "rank": RANK,
    "method": "DoRA" if USE_DORA else "LoRA",
    "seed": SEED,
    "train_loss": train_result.training_loss,
    "execution_accuracy": acc,
}

file_exists = os.path.exists(RESULTS_CSV)
with open(RESULTS_CSV, "a", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=row.keys())
    if not file_exists:
        writer.writeheader()
    writer.writerow(row)

print(row)